# 2장 09. 평가 기록과 MLflow Tracking 기초 실습

이 notebook은 평가 결과를 나중에 다시 비교할 수 있게 **기록으로 남기는 방법**을 확인합니다. MLflow를 처음 보는 사람은 먼저 “실험 기록장”이라고 생각하면 됩니다.


### 따라하기 안내: 평가 기록 확인 흐름

이 notebook은 “평가를 다시 실행할 수 있는가?”보다 “나중에 같은 조건으로 비교할 수 있게 기록이 남았는가?”에 초점을 둡니다.

1. 평가 기록이 어느 repo와 파일 경로에 저장되는지 확인합니다.
2. test 데이터, feature, label, threshold 조건을 먼저 봅니다.
3. 평가를 실행하고 metric과 FP/FN을 확인합니다.
4. 저장된 JSON 기록을 다시 읽어 params와 metrics가 함께 남았는지 확인합니다.
5. validation 비교 artifact와 test 평가 기록의 조건이 이어지는지 봅니다.
6. MLflow가 있으면 tracking 상태를 확인하고, 없으면 JSON artifact를 필수 근거로 사용합니다.

출력에서 가장 중요한 것은 “지표 숫자”만이 아니라 “그 숫자를 만든 조건”입니다.


## 0. 평가 기록을 처음 보는 사람을 위한 기초 예제

MLflow를 배우기 전에 먼저 “평가 기록” 자체를 이해해야 합니다. 평가 기록은 단순히 metric 숫자를 저장하는 파일이 아닙니다.

좋은 평가 기록에는 최소한 세 가지가 함께 있어야 합니다.

| 구분 | 쉬운 뜻 | 예시 |
| --- | --- | --- |
| params | 결과를 만든 조건 | model_version, threshold, dataset |
| metrics | 평가 결과 숫자 | precision, recall, FP, FN |
| artifacts | 같이 보관할 파일 | JSON report, model file path |

MLflow는 이 구조를 검색하고 비교하기 쉽게 해 주는 도구입니다. 이번 강의에서는 MLflow 서버 운영보다 **무엇을 기록해야 나중에 비교 가능한가**에 집중합니다.


### 0-1. 작은 평가 기록 dict 만들기

먼저 MLflow 없이 Python dict로 평가 기록 구조를 만들어 봅니다. 실제 MLflow도 결국 params, metrics, artifacts를 함께 관리한다는 점에서 같은 생각입니다.


In [ ]:
import pandas as pd

tracking_basic_record = {
    "run_name": "toy_model_eval",
    "params": {
        "dataset": "toy_test",
        "model_version": "demo-v1",
        "threshold": 0.50,
        "positive_label": "high_risk",
    },
    "metrics": {
        "precision": 0.75,
        "recall": 0.60,
        "FP": 1,
        "FN": 2,
    },
    "artifacts": {
        "evaluation_json": "artifacts/experiments/chapter_02/model_test_eval.json",
    },
}

tracking_basic_sections = pd.DataFrame(
    [
        {"section": "params", "easy_meaning": "평가 조건", "example_keys": ", ".join(tracking_basic_record["params"].keys())},
        {"section": "metrics", "easy_meaning": "평가 결과", "example_keys": ", ".join(tracking_basic_record["metrics"].keys())},
        {"section": "artifacts", "easy_meaning": "함께 보관할 파일", "example_keys": ", ".join(tracking_basic_record["artifacts"].keys())},
    ]
)

display(tracking_basic_sections)


**출력 확인**

평가 기록은 `metrics`만 있으면 부족합니다. `threshold`나 `dataset` 같은 조건이 없으면 나중에 숫자를 비교하기 어렵습니다.

예를 들어 recall이 달라졌을 때, 모델이 바뀐 것인지 threshold가 바뀐 것인지 알 수 없게 됩니다.


### 0-2. 기록을 표로 펼쳐 보기

JSON이나 MLflow run은 사람이 바로 읽기 어려울 수 있습니다. 그래서 notebook에서는 params와 metrics를 표로 펼쳐서 확인합니다.


In [ ]:
tracking_basic_params = pd.DataFrame(
    [{"key": key, "value": value} for key, value in tracking_basic_record["params"].items()]
)
tracking_basic_metrics = pd.DataFrame(
    [{"metric": key, "value": value} for key, value in tracking_basic_record["metrics"].items()]
)
tracking_basic_artifacts = pd.DataFrame(
    [{"artifact": key, "path": value} for key, value in tracking_basic_record["artifacts"].items()]
)


**출력 확인: `tracking_basic_params`**

params는 평가 조건입니다. 어떤 데이터, 어떤 모델, 어떤 threshold에서 나온 결과인지 확인합니다.


In [ ]:
display(tracking_basic_params)

**출력 확인: `tracking_basic_metrics`**

metrics는 평가 결과 숫자입니다. 숫자만 보지 말고 앞의 params와 함께 읽어야 합니다.


In [ ]:
display(tracking_basic_metrics)

**출력 확인: `tracking_basic_artifacts`**

artifacts는 나중에 다시 확인할 파일 근거입니다. JSON 기록이나 모델 파일 경로가 여기에 해당합니다.


In [ ]:
display(tracking_basic_artifacts)

**출력 확인**

- params 표는 “어떤 조건에서 평가했는가”를 보여 줍니다.
- metrics 표는 “그 조건에서 결과가 어땠는가”를 보여 줍니다.
- artifacts 표는 “나중에 다시 확인할 파일이 어디 있는가”를 보여 줍니다.

본 실습에서는 같은 구조를 `model_test_eval.json`과 선택적 MLflow 기록에서 확인합니다.


## 1. 실행 환경과 기록 범위

### 1-1. 일반 notebook 실행 기준 고정

이 셀에서는 평가 기록을 어느 repo, 어느 데이터, 어느 artifact 경로 기준으로 재생성하는지 고정하는 것입니다. 이 notebook은 Lite용 읽기 전용 실습이 아니라, 로컬 artifact를 다시 쓸 수 있는 일반 실행 경로입니다.

실행 결과는 `artifacts/experiments/chapter_02/model_test_eval.json`을 갱신합니다. MLflow가 설치되어 있으면 같은 실행 조건과 지표가 `artifacts/mlflow.db`에도 남습니다.

이 단계에서는 아직 metric을 계산하지 않습니다. 먼저 repo root, package path, 입력 데이터, 출력 artifact의 역할을 명확히 합니다.

### 따라하기 안내: 기록 환경 준비

**이 셀에서 하는 일**  
실습에 필요한 도구를 불러옵니다.

**확인할 점**  
패키지 이름, 기준 label, threshold처럼 뒤 셀에서 계속 사용할 기준값을 확인합니다.

**왜 중요한가**  
처음 기준이 잘못 잡히면 뒤의 모든 출력도 잘못 해석됩니다.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd


def find_repo_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "packages" / "ai-quality" / "src").exists() and (base / "data").exists():
            return base
    raise RuntimeError("repo root를 찾지 못했습니다.")


ROOT = find_repo_root()
PACKAGE_SRC = ROOT / "packages" / "ai-quality" / "src"
if str(PACKAGE_SRC) not in sys.path:
    sys.path.insert(0, str(PACKAGE_SRC))

TEST_DATASET_PATH = ROOT / "data" / "vital_signs_test.csv"
MODEL_TEST_EVAL_PATH = ROOT / "artifacts" / "experiments" / "chapter_02" / "model_test_eval.json"
COMPARISON_ARTIFACT_PATH = ROOT / "artifacts" / "experiments" / "chapter_02" / "validation_degradation_comparison.json"
MLFLOW_DB_PATH = ROOT / "artifacts" / "mlflow.db"

execution_contract = pd.DataFrame(
    [
        {"item": "repo_root", "value": str(ROOT), "qa_use": "artifact 경로를 감사 가능하게 남깁니다."},
        {"item": "test_dataset", "value": str(TEST_DATASET_PATH), "qa_use": "선택된 모델 평가 데이터를 고정합니다."},
        {"item": "json_record", "value": str(MODEL_TEST_EVAL_PATH), "qa_use": "MLflow가 없어도 확인할 필수 평가 기록입니다."},
        {"item": "mlflow_db", "value": str(MLFLOW_DB_PATH), "qa_use": "MLflow가 설치된 경우 선택적으로 남는 tracking DB입니다."},
        {"item": "runtime_role", "value": "general_notebook_record_regeneration", "qa_use": "Lite artifact inspection과 구분합니다."},
    ]
)

display(execution_contract)

이 출력에서 확인할 핵심은 이 notebook이 평가 기록을 다시 쓰는 일반 실행 경로라는 점입니다. 준비된 artifact를 읽는 것과 직접 재생성한 것을 보고서에서 구분해야 재현 범위가 명확해집니다.

## 2. test 평가 데이터와 실행 조건 brief

### 2-1. 평가 데이터가 metric에 들어가기 전 상태 확인

이 셀에서는 test metric이 어떤 데이터 조건에서 계산될지 먼저 확인하는 것입니다. 지표가 나온 뒤에 데이터셋 이름만 확인하면, row count, label support, feature 결측 같은 전제가 빠질 수 있습니다.

한 row는 선택된 모델을 평가하는 classification sample 하나입니다. 이 데이터는 실제 의료 확인이 아니라 생체신호 기반 위험 알림 시스템의 AI QA 예제입니다.

이 단계에서는 metric을 계산하지 않습니다. 먼저 feature 목록, label column, threshold, 모델 artifact, label 분포, feature readiness를 보여주어 기록해야 할 전제를 고정합니다.

### 따라하기 안내: 평가 조건 준비

**이 셀에서 하는 일**  
평가에 사용할 feature 목록, label 컬럼, threshold, 모델 파일 경로를 확인합니다.

**확인할 점**  
평가 조건이 하나의 표로 남는지 봅니다. 특히 모델 파일 존재 여부와 threshold를 확인합니다.

**왜 중요한가**  
평가 기록은 metric만 저장하면 부족합니다. 어떤 조건에서 나온 metric인지 함께 남아야 나중에 비교할 수 있습니다.


In [ ]:
from ai_quality.labs.ch02_model_quality import (
    chapter_model_path,
    feature_columns,
    operating_threshold,
    target_column,
)
from ai_quality.model_quality.infrastructure.mlflow_tracker import mlflow_available

features = feature_columns()
target = target_column()
threshold = operating_threshold()
model_path = chapter_model_path()

test_dataframe = pd.read_csv(TEST_DATASET_PATH)
preview_columns = [column for column in ["patient_id", "timestamp", *features, target] if column in test_dataframe.columns]

raw_data_brief = pd.DataFrame(
    [
        {"check": "dataset_path", "observed": str(TEST_DATASET_PATH), "qa_use": "평가 대상 파일을 고정합니다."},
        {"check": "row_grain", "observed": "one model test classification sample", "qa_use": "metric 분모를 설명합니다."},
        {"check": "row_count", "observed": test_dataframe.shape[0], "qa_use": "FP/FN과 metric 계산 기준입니다."},
        {"check": "column_count", "observed": test_dataframe.shape[1], "qa_use": "feature/label 누락 여부를 보기 전 구조입니다."},
        {"check": "feature_count", "observed": len(features), "qa_use": "모델 입력 조건을 기록합니다."},
        {"check": "operating_threshold", "observed": threshold, "qa_use": "score를 prediction으로 바꾸는 기준입니다."},
        {"check": "model_artifact_exists", "observed": model_path.exists(), "qa_use": "같은 모델로 재현 가능한지 확인합니다."},
        {"check": "mlflow_available", "observed": mlflow_available(), "qa_use": "선택 tracking 기록 생성 가능 여부입니다."},
    ]
)
label_distribution = (
    test_dataframe[target]
    .value_counts(dropna=False)
    .rename("count")
    .reset_index()
    .rename(columns={"index": "label"})
    .assign(ratio_pct=lambda table: (table["count"] / len(test_dataframe) * 100).round(2))
)
feature_readiness = (
    test_dataframe.loc[:, features]
    .apply(pd.to_numeric, errors="coerce")
    .isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "feature"})
    .assign(missing_ratio_pct=lambda table: (table["missing_count"] / len(test_dataframe) * 100).round(2))
    .sort_values(["missing_count", "feature"], ascending=[False, True])
)


### 출력 확인: `raw_data_brief`

**확인할 점**  
행 수, 컬럼 수, 데이터셋 이름을 봅니다.

**왜 중요한가**  
분석 대상이 기대와 다르면 뒤의 결측, label, metric 결과도 의미가 달라집니다.

**다음 단계**  
대상이 맞으면 preview와 label 분포로 넘어갑니다.


In [ ]:
display(raw_data_brief)


### 출력 확인: `test_dataframe.loc[:, preview_columns].head()`

**확인할 점**  
몇 개 행을 직접 보고 컬럼 이름과 값 모양이 예상과 맞는지 봅니다.

**왜 중요한가**  
Preview는 전체 확인이 아니라 데이터 모양 확인입니다.

**다음 단계**  
분포나 metric은 뒤의 요약 표에서 확인합니다.


In [ ]:
display(test_dataframe.loc[:, preview_columns].head())


### 출력 확인: `label_distribution`

**확인할 점**  
`high_risk`와 `low_risk`의 개수와 비율을 봅니다.

**왜 중요한가**  
`high_risk`는 관심 class입니다. 이 수가 적으면 recall 같은 지표가 크게 흔들릴 수 있습니다.

**다음 단계**  
두 class가 모두 있는지 확인한 뒤 label gate로 넘어갑니다.


In [ ]:
display(label_distribution)


### 출력 확인: `feature_readiness`

**확인할 점**  
feature별 누락, 결측, invalid 상태를 봅니다.

**왜 중요한가**  
문제가 있는 feature는 score와 metric 변화의 원인 후보가 됩니다.

**다음 단계**  
문제가 있는 feature 이름을 기억하고 score/metric 변화와 연결합니다.

**주의할 점**  
바로 모델 결함으로 단정하지 않습니다.


In [ ]:
display(feature_readiness)


이 출력에서 확인할 핵심은 metric이 계산되기 전에 평가 조건이 먼저 고정된다는 점입니다. `feature_count`, `operating_threshold`, `model_artifact_exists`는 이후 JSON과 MLflow 기록에 반드시 남아야 하는 비교 전제입니다.

label 분포는 모델 성능을 좋게 보이게 만들기 위한 정보가 아닙니다. 관심 클래스 표본 수가 달라지면 Precision/Recall 해석도 달라지므로, 지표를 읽기 전에 분모와 class support를 먼저 확인합니다.

### 2-2. 기록해야 할 조건과 metric 역할 분리

이 셀에서는 평가 기록의 `params`와 `metrics`가 서로 다른 역할을 갖는다는 점을 확인하는 것입니다. `params`는 무엇으로 평가했는가를 설명하고, `metrics`는 그 조건에서 어떤 결과가 나왔는가를 설명합니다.

QA는 지표를 먼저 비교하지 않습니다. 데이터셋 버전, feature 목록, label mapping, model version, threshold가 같거나 의도적으로 달라졌는지 확인한 뒤 metric을 비교합니다.

이 표에서는 실행 전에 반드시 남겨야 할 기록 필드를 확인하는 것입니다. 이 필드가 빠지면 나중에 품질 회귀가 모델 변경 때문인지, 데이터 변경 때문인지, threshold 변경 때문인지 분리하기 어렵습니다.

### 따라하기 안내: 필수 field 확인

**이 셀에서 하는 일**  
평가 기록에 반드시 들어가야 하는 필드를 미리 정리합니다.

**확인할 점**  
`params`, `metrics`, `artifact path`처럼 나중에 비교에 필요한 항목이 빠지지 않았는지 봅니다.

**왜 중요한가**  
기록 필드가 빠지면 나중에 지표 차이가 모델 때문인지 데이터 때문인지 확인하기 어렵습니다.


In [ ]:
required_record_fields = pd.DataFrame(
    [
        {"record_area": "params", "field": "dataset_name", "qa_question": "같은 평가 데이터셋인가", "required": True},
        {"record_area": "params", "field": "dataset_version", "qa_question": "비교 가능한 데이터 버전인가", "required": True},
        {"record_area": "params", "field": "feature_columns", "qa_question": "같은 입력 feature인가", "required": True},
        {"record_area": "params", "field": "label_mapping", "qa_question": "같은 label 표준화 기준인가", "required": True},
        {"record_area": "params", "field": "model_name", "qa_question": "같은 모델 계열인가", "required": True},
        {"record_area": "params", "field": "model_version", "qa_question": "같은 모델 버전인가", "required": True},
        {"record_area": "params", "field": "operating_threshold", "qa_question": "같은 prediction 변환 기준인가", "required": True},
        {"record_area": "metrics", "field": "precision", "qa_question": "관심 클래스 예측 신뢰도가 어떤가", "required": True},
        {"record_area": "metrics", "field": "recall", "qa_question": "관심 클래스 탐지 누락이 어떤가", "required": True},
        {"record_area": "metrics", "field": "false_positive", "qa_question": "오탐 부담이 몇 건인가", "required": True},
        {"record_area": "metrics", "field": "false_negative", "qa_question": "미탐 부담이 몇 건인가", "required": True},
    ]
)

display(required_record_fields)

이 표에서 확인할 핵심은 MLflow나 JSON이 단순 저장소가 아니라 비교 조건을 보존하는 장치라는 점입니다. 수강생은 도구 화면보다 기록 필드의 완전성을 먼저 봐야 합니다.

## 3. 평가 기록 생성과 JSON 감사

### 3-1. 같은 실행에서 metric과 기록 경로 생성

이 셀에서는 test 평가가 metric과 artifact 경로를 한 번의 실행으로 함께 남기는지 확인하는 것입니다. 콘솔에 metric만 출력되면 나중에 reviewer가 같은 조건을 다시 확인하기 어렵습니다.

`record_model_test_evaluation()`은 기준선 모델을 로드하고, `vital_signs_test.csv`에 대해 score를 만든 뒤, 운영 threshold `0.50`으로 prediction을 계산해 metric을 기록합니다. 같은 실행에서 JSON artifact를 만들고, MLflow가 설치되어 있으면 local tracking DB에도 남깁니다.

이 실습의 목적은 모델 성능 향상이 아닙니다. 선택된 모델과 threshold의 test 평가 조건이 audit-ready하게 기록되는지 확인하는 것입니다.

### 따라하기 안내: 평가 기록 생성

**이 셀에서 하는 일**  
test 데이터로 평가를 실행하고 JSON 평가 기록을 만듭니다. MLflow가 설치된 환경이면 선택적으로 tracking 기록도 남깁니다.

**확인할 점**  
metric과 FP/FN이 계산되고, 기록 파일 경로가 만들어졌는지 봅니다.

**왜 중요한가**  
이 셀이 이번 notebook의 핵심입니다. 계산 결과가 파일로 남아야 나중에 같은 조건으로 다시 비교할 수 있습니다.


In [ ]:
from ai_quality.labs.ch02_model_quality import record_model_test_evaluation

record_result = record_model_test_evaluation()
report = record_result.report

metric_summary = pd.DataFrame(
    [
        {"metric": "accuracy", "value": round(report.metrics.accuracy, 4), "qa_use": "전체 정답률 참고값입니다. 단독 확인 기준으로 쓰지 않습니다."},
        {"metric": "precision", "value": round(report.metrics.precision, 4), "qa_use": "관심 클래스로 예측한 sample의 신뢰도를 봅니다."},
        {"metric": "recall", "value": round(report.metrics.recall, 4), "qa_use": "관심 클래스 sample 중 놓친 비율을 봅니다."},
        {"metric": "f1_score", "value": round(report.metrics.f1_score, 4), "qa_use": "Precision/Recall 균형 참고값입니다."},
        {"metric": "auroc", "value": round(report.metrics.auroc or 0.0, 4), "qa_use": "score 구분력 참고값입니다."},
        {"metric": "pr_auc", "value": round(report.metrics.pr_auc or 0.0, 4), "qa_use": "관심 클래스 중심 score 품질 참고값입니다."},
    ]
)
confusion_summary = pd.DataFrame(
    [
        {"error_or_hit": "true_positive", "count": report.confusion_matrix.true_positive, "qa_use": "관심 클래스를 맞힌 건수입니다."},
        {"error_or_hit": "false_positive", "count": report.confusion_matrix.false_positive, "qa_use": "low_risk sample을 high_risk로 잘못 예측한 건수입니다."},
        {"error_or_hit": "false_negative", "count": report.confusion_matrix.false_negative, "qa_use": "high_risk sample을 low_risk로 놓친 건수입니다."},
        {"error_or_hit": "true_negative", "count": report.confusion_matrix.true_negative, "qa_use": "low_risk sample을 맞힌 건수입니다."},
    ]
)
record_paths = pd.DataFrame(
    [
        {"artifact": "json_experiment_record", "path": str(record_result.json_path), "exists": record_result.json_path.exists(), "qa_use": "필수 평가 기록입니다."},
        {"artifact": "mlflow_tracking_db", "path": str(record_result.mlflow_path) if record_result.mlflow_path else str(MLFLOW_DB_PATH), "exists": bool(record_result.mlflow_path and record_result.mlflow_path.exists()), "qa_use": "설치된 경우 선택 tracking 기록입니다."},
    ]
)


### 출력 확인: `metric_summary`

**확인할 점**  
accuracy, precision, recall, F1, AUROC/PR-AUC를 봅니다.

**왜 중요한가**  
precision은 FP의 영향을 받고, recall은 FN의 영향을 받습니다. accuracy 하나로는 오류 유형을 알 수 없습니다.

**다음 단계**  
confusion matrix의 FP/FN과 함께 해석합니다.

**주의할 점**  
metric에는 데이터셋과 threshold 조건을 같이 붙여야 합니다.


In [ ]:
display(metric_summary)


### 출력 확인: `confusion_summary`

**확인할 점**  
TP, FP, FN, TN을 봅니다. 특히 FP와 FN을 구분합니다.

**왜 중요한가**  
FP는 low risk를 high risk로 잘못 알린 경우이고, FN은 high risk를 놓친 경우입니다. precision과 recall은 이 오류와 연결됩니다.

**다음 단계**  
FP/FN 중 어떤 오류가 더 큰지 metric table과 함께 봅니다.

**주의할 점**  
actual과 prediction 축을 바꾸어 읽으면 해석이 반대로 됩니다.


In [ ]:
display(confusion_summary)


### 출력 확인: `record_paths`

**확인할 점**  
생성된 파일 경로와 다시 읽기 가능 여부를 봅니다.

**왜 중요한가**  
파일로 남아야 나중에 같은 근거를 다시 확인할 수 있습니다.

**다음 단계**  
경로를 확인한 뒤 파일 내용의 핵심 field를 봅니다.


In [ ]:
display(record_paths)


이 출력에서 확인할 핵심은 Precision `1.0000`만 보고 좋은 모델이라고 결론 내리지 않는 것입니다. FP가 `0`이어도 FN이 많이 남으면 관심 클래스 누락 위험이 큽니다.

MLflow tracking DB가 없더라도 실습 실패가 아닙니다. 이번 장의 필수 증거는 JSON 평가 기록이며, MLflow는 같은 기록을 추적 도구에서도 볼 수 있는 선택 경로입니다.

### 3-2. JSON 기록이 비교 조건과 metric을 함께 담았는지 확인

이 셀에서는 생성된 JSON artifact가 실행 조건과 metric을 분리해서 보존하는지 확인하는 것입니다. 파일이 존재하는 것만으로는 충분하지 않고, 나중에 비교할 필드가 빠지지 않았는지 봐야 합니다.

`params`에는 데이터셋, 모델, feature, label, threshold가 들어가야 합니다. `metrics`에는 Accuracy 하나가 아니라 Precision, Recall, FP, FN, PR-AUC 같은 확인 지표가 함께 있어야 합니다.

이 검사는 MLflow UI를 열기 전에도 수행할 수 있는 최소 감사 절차입니다.

### 따라하기 안내: model test record 읽기

**이 셀에서 하는 일**  
저장된 model test 평가 기록을 읽습니다.

**확인할 점**  
데이터셋, model version, threshold, metric을 확인합니다.

**왜 중요한가**  
metric만 있으면 재현이 어렵습니다. 조건이 같이 남아야 합니다.


In [ ]:
model_test_record = json.loads(MODEL_TEST_EVAL_PATH.read_text(encoding="utf-8"))

params_table = pd.DataFrame(
    [
        {"field": key, "value": value, "qa_use": "평가 조건 확인"}
        for key, value in model_test_record["params"].items()
    ]
)
metrics_table = pd.DataFrame(
    [
        {"metric": key, "value": round(float(value), 4), "qa_use": "평가 결과 확인"}
        for key, value in model_test_record["metrics"].items()
    ]
)
field_completeness = required_record_fields.assign(
    present=lambda table: table.apply(
        lambda row: row["field"] in model_test_record.get(row["record_area"], {}),
        axis=1,
    ),
    qa_status=lambda table: table["present"].map({True: "pass", False: "review"}),
)
artifact_provenance = pd.DataFrame(
    [
        {"artifact": path, "exists": Path(path).exists(), "qa_use": "평가 재현에 필요한 모델 파일 추적"}
        for path in model_test_record.get("artifacts", [])
    ]
)


### 출력 확인: `params_table`

**확인할 점**  
평가 기록에 필요한 조건과 field가 남아 있는지 봅니다.

**왜 중요한가**  
metric만 저장하면 나중에 재현하거나 비교하기 어렵습니다.

**다음 단계**  
dataset, model version, threshold, feature 기준을 함께 확인합니다.

**주의할 점**  
MLflow가 없어도 JSON artifact에 필수 조건이 남아 있으면 됩니다.


In [ ]:
display(params_table)


### 출력 확인: `metrics_table`

**확인할 점**  
accuracy, precision, recall, F1, AUROC/PR-AUC를 봅니다.

**왜 중요한가**  
precision은 FP의 영향을 받고, recall은 FN의 영향을 받습니다. accuracy 하나로는 오류 유형을 알 수 없습니다.

**다음 단계**  
confusion matrix의 FP/FN과 함께 해석합니다.

**주의할 점**  
metric에는 데이터셋과 threshold 조건을 같이 붙여야 합니다.


In [ ]:
display(metrics_table)


### 출력 확인: `field_completeness`

**확인할 점**  
평가 기록에 필요한 조건과 field가 남아 있는지 봅니다.

**왜 중요한가**  
metric만 저장하면 나중에 재현하거나 비교하기 어렵습니다.

**다음 단계**  
dataset, model version, threshold, feature 기준을 함께 확인합니다.

**주의할 점**  
MLflow가 없어도 JSON artifact에 필수 조건이 남아 있으면 됩니다.


In [ ]:
display(field_completeness)


### 출력 확인: `artifact_provenance`

**확인할 점**  
데이터나 artifact가 어디에서 왔는지 봅니다. 경로와 실행 범위를 먼저 확인합니다.

**왜 중요한가**  
같은 숫자라도 prepared artifact인지 local 재생성인지에 따라 보고서 표현이 달라집니다.

**다음 단계**  
이 범위를 기억한 상태로 다음 표를 읽습니다.


In [ ]:
display(artifact_provenance)


이 출력에서 확인할 핵심은 `params`와 `metrics`가 같은 run 안에 함께 있다는 점입니다. 이 구조가 있어야 나중에 새 모델 또는 새 데이터 결과와 비교할 때 조건 차이를 먼저 분리할 수 있습니다.

`field_completeness`에 `review`가 나오면 metric 해석을 바로 진행하지 않습니다. 누락 필드의 owner를 확인하고, 비교 조건이 재현 가능한지 먼저 정리합니다.

## 4. validation 비교 artifact와 회귀 확인 연결

### 4-1. 단일 test 기록과 validation 비교 기록을 분리해서 읽기

이 셀에서는 `model_test_eval.json`과 `validation_degradation_comparison.json`을 섞지 않고 읽는 것입니다. test 기록은 선택된 모델과 threshold의 기준 평가이고, validation 비교 artifact는 같은 모델/threshold에서 데이터 품질 저하가 지표에 어떤 변화를 만들었는지 보여줍니다.

두 기록은 서로 다른 질문에 답합니다. test 기록은 “선택한 모델이 어떤 조건에서 어떤 기준 지표를 냈는가”를 말하고, validation 비교는 “데이터 조건이 흔들렸을 때 같은 모델과 threshold에서 어떤 변화가 생겼는가”를 말합니다.

같은 모델 버전과 threshold가 유지되면, 품질 저하 validation의 FP 증가와 Precision 하락은 입력 품질 저하를 강한 원인 후보로 남길 수 있습니다.

### 따라하기 안내: comparison record 읽기

**이 셀에서 하는 일**  
validation 비교 기록을 읽습니다.

**확인할 점**  
baseline/degraded metric과 조건을 확인합니다.

**왜 중요한가**  
test 평가와 validation 비교는 서로 다른 목적의 근거입니다.


In [ ]:
comparison_record = json.loads(COMPARISON_ARTIFACT_PATH.read_text(encoding="utf-8"))
comparison_model = comparison_record["model"]
comparison_datasets = comparison_record["datasets"]
comparison_deltas = comparison_record["deltas"]

condition_alignment = pd.DataFrame(
    [
        {
            "condition": "model_name",
            "model_test_eval": model_test_record["params"].get("model_name"),
            "validation_comparison": comparison_model.get("model_name"),
            "qa_status": "pass" if model_test_record["params"].get("model_name") == comparison_model.get("model_name") else "review",
        },
        {
            "condition": "model_version",
            "model_test_eval": model_test_record["params"].get("model_version"),
            "validation_comparison": comparison_model.get("model_version"),
            "qa_status": "pass" if model_test_record["params"].get("model_version") == comparison_model.get("model_version") else "review",
        },
        {
            "condition": "operating_threshold",
            "model_test_eval": float(model_test_record["params"].get("operating_threshold")),
            "validation_comparison": float(comparison_model.get("operating_threshold")),
            "qa_status": "pass" if float(model_test_record["params"].get("operating_threshold")) == float(comparison_model.get("operating_threshold")) else "review",
        },
        {
            "condition": "feature_columns",
            "model_test_eval": model_test_record["params"].get("feature_columns"),
            "validation_comparison": ",".join(comparison_model.get("feature_columns", [])),
            "qa_status": "pass" if model_test_record["params"].get("feature_columns") == ",".join(comparison_model.get("feature_columns", [])) else "review",
        },
    ]
)
validation_metric_comparison = pd.DataFrame(
    [
        {
            "metric_or_error": "precision",
            "valid_baseline": comparison_datasets["valid_baseline"]["metrics"]["precision"],
            "valid_degraded": comparison_datasets["valid_degraded"]["metrics"]["precision"],
            "delta": comparison_deltas["precision"],
            "qa_interpretation": "관심 클래스 예측 신뢰도 하락 후보",
        },
        {
            "metric_or_error": "recall",
            "valid_baseline": comparison_datasets["valid_baseline"]["metrics"]["recall"],
            "valid_degraded": comparison_datasets["valid_degraded"]["metrics"]["recall"],
            "delta": comparison_deltas["recall"],
            "qa_interpretation": "관심 클래스 누락 변화 후보",
        },
        {
            "metric_or_error": "false_positive",
            "valid_baseline": comparison_datasets["valid_baseline"]["confusion_matrix"]["false_positive"],
            "valid_degraded": comparison_datasets["valid_degraded"]["confusion_matrix"]["false_positive"],
            "delta": comparison_deltas["false_positive"],
            "qa_interpretation": "오탐 운영 부담 증가 후보",
        },
        {
            "metric_or_error": "false_negative",
            "valid_baseline": comparison_datasets["valid_baseline"]["confusion_matrix"]["false_negative"],
            "valid_degraded": comparison_datasets["valid_degraded"]["confusion_matrix"]["false_negative"],
            "delta": comparison_deltas["false_negative"],
            "qa_interpretation": "미탐 운영 위험 증가 후보",
        },
    ]
)


### 출력 확인: `condition_alignment`

**확인할 점**  
평가 기록에 필요한 조건과 field가 남아 있는지 봅니다.

**왜 중요한가**  
metric만 저장하면 나중에 재현하거나 비교하기 어렵습니다.

**다음 단계**  
dataset, model version, threshold, feature 기준을 함께 확인합니다.

**주의할 점**  
MLflow가 없어도 JSON artifact에 필수 조건이 남아 있으면 됩니다.


In [ ]:
display(condition_alignment)


### 출력 확인: `validation_metric_comparison`

**확인할 점**  
accuracy, precision, recall, F1, AUROC/PR-AUC를 봅니다.

**왜 중요한가**  
precision은 FP의 영향을 받고, recall은 FN의 영향을 받습니다. accuracy 하나로는 오류 유형을 알 수 없습니다.

**다음 단계**  
confusion matrix의 FP/FN과 함께 해석합니다.

**주의할 점**  
metric에는 데이터셋과 threshold 조건을 같이 붙여야 합니다.


In [ ]:
display(validation_metric_comparison)


이 출력에서 확인할 핵심은 validation 비교의 모델 버전과 threshold가 test 평가 기록과 같은 계열로 추적된다는 점입니다. 조건이 맞으면 지표 변화 원인을 데이터 품질 저하 쪽으로 좁힐 수 있고, 조건이 다르면 모델 또는 threshold 변경 가능성을 먼저 분리해야 합니다.

Precision 하락과 FP 증가는 모델이 갑자기 나빠졌다고 단정할 근거가 아닙니다. 2-2 GE 결과와 2-5 데이터-지표 연결 결과를 함께 보아 feature 품질 실패가 같은 방향의 원인 후보인지 확인해야 합니다.

### 4-2. MLflow 선택 경로를 QA 관점으로 해석

이 셀에서는 MLflow 사용 가능 여부를 실습 성공/실패로 보지 않고, 같은 기록이 추적 도구에도 남았는지 확인하는 것입니다. MLflow는 기록을 검색하고 비교하는 도구이며, 이번 장의 필수 근거는 JSON artifact입니다.

MLflow가 설치되어 있으면 `record_model_test_evaluation()` 실행 중 local tracking DB에 같은 params와 metrics가 기록됩니다. 설치되어 있지 않으면 `mlflow_status`는 `optional_unavailable`로 남기고 JSON 기록을 근거로 진행합니다.

이 구분이 있어야 교육 환경마다 MLflow 설치 여부가 달라도 QA 해석 흐름이 깨지지 않습니다.

### 따라하기 안내: MLflow 상태 확인

**이 셀에서 하는 일**  
현재 환경에서 MLflow tracking을 사용할 수 있는지 확인합니다.

**확인할 점**  
MLflow가 없어도 JSON 기록이 남아 있는지, MLflow가 있으면 tracking DB 위치가 어디인지 봅니다.

**왜 중요한가**  
이번 강의에서 MLflow는 필수 실행 조건이 아닙니다. 필수 근거는 JSON 평가 기록이고, MLflow는 검색과 비교를 돕는 선택 도구입니다.


In [ ]:
mlflow_status = pd.DataFrame(
    [
        {
            "check": "mlflow_package_available",
            "observed": mlflow_available(),
            "qa_status": "available" if mlflow_available() else "optional_unavailable",
            "qa_interpretation": "설치되어 있으면 tracking DB 확인까지 진행합니다. 없으면 JSON artifact로 필수 확인을 완료합니다.",
        },
        {
            "check": "mlflow_path_returned",
            "observed": str(record_result.mlflow_path) if record_result.mlflow_path else None,
            "qa_status": "recorded" if record_result.mlflow_path else "not_recorded",
            "qa_interpretation": "선택 tracking 산출물의 존재 여부입니다.",
        },
        {
            "check": "json_record_still_exists",
            "observed": MODEL_TEST_EVAL_PATH.exists(),
            "qa_status": "pass" if MODEL_TEST_EVAL_PATH.exists() else "review",
            "qa_interpretation": "필수 평가 기록이 남아 있어야 합니다.",
        },
    ]
)

tracking_handoff = pd.DataFrame(
    [
        {"next_check": "MLflow UI", "when": "mlflow_available=True", "qa_question": "run 목록에서 `model_test_eval`의 params/metrics/artifacts가 같은지 확인합니다."},
        {"next_check": "JSON artifact", "when": "always", "qa_question": "MLflow가 없어도 평가 조건과 FP/FN이 남아 있는지 확인합니다."},
        {"next_check": "Serving API", "when": "3장", "qa_question": "API 응답의 `model_version`, `threshold`, `prediction`이 평가 기록과 일치하는지 확인합니다."},
    ]
)


### 출력 확인: `mlflow_status`

**확인할 점**  
평가 기록에 필요한 조건과 field가 남아 있는지 봅니다.

**왜 중요한가**  
metric만 저장하면 나중에 재현하거나 비교하기 어렵습니다.

**다음 단계**  
dataset, model version, threshold, feature 기준을 함께 확인합니다.

**주의할 점**  
MLflow가 없어도 JSON artifact에 필수 조건이 남아 있으면 됩니다.


In [ ]:
display(mlflow_status)


### 출력 확인: `tracking_handoff`

**확인할 점**  
평가 기록에 필요한 조건과 field가 남아 있는지 봅니다.

**왜 중요한가**  
metric만 저장하면 나중에 재현하거나 비교하기 어렵습니다.

**다음 단계**  
dataset, model version, threshold, feature 기준을 함께 확인합니다.

**주의할 점**  
MLflow가 없어도 JSON artifact에 필수 조건이 남아 있으면 됩니다.


In [ ]:
display(tracking_handoff)


이 출력에서 확인할 핵심은 MLflow가 없더라도 수업의 해석 근거가 사라지지 않는다는 점입니다. JSON artifact는 필수 증거이고, MLflow는 운영 환경에서 검색과 비교를 쉽게 만드는 선택 도구입니다.

## 5. QA 해석과 다음 장 handoff

### 5-1. 기록 완전성을 보고 문장으로 바꾸기

마지막 정리는 평가 기록을 “저장했다”가 아니라 “비교 가능한 조건과 결과가 함께 남았다”로 표현하는 것입니다. QA 보고 문장은 데이터셋, 모델 버전, threshold, Precision/Recall, FP/FN, 남은 확인 항목을 함께 담아야 합니다.

2장의 결론은 모델 지표를 단독 숫자로 보지 않는 것입니다. 데이터 검증 결과, 평가 조건, 모델 버전, threshold, 오류 유형이 함께 남아야 3장에서 API 응답과 운영 로그를 확인할 기준이 생깁니다.

이 출력은 3장 serving notebook으로 넘길 handoff입니다. 3장에서는 외부 API 또는 로컬 API 응답이 같은 `model_version`, `threshold`, `score`, `prediction` 구조를 유지하는지 확인합니다.

### 따라하기 안내: 기록 준비 상태

**이 셀에서 하는 일**  
평가 기록, validation 비교 기록, MLflow 상태를 한 번에 점검합니다.

**확인할 점**  
최종 보고서에 쓸 근거 파일이 준비되었는지, 어떤 리스크가 남았는지 봅니다.

**왜 중요한가**  
마지막 단계에서는 새 계산을 더 하는 것이 아니라, 제출 가능한 기록 상태인지 확인합니다.


In [ ]:
record_ready = bool(field_completeness["present"].all() and MODEL_TEST_EVAL_PATH.exists())
condition_ready = bool(condition_alignment["qa_status"].eq("pass").all())

chapter_02_tracking_packet = pd.DataFrame(
    [
        {
            "evidence": "model_test_eval_record",
            "observed": f"dataset_version={model_test_record['params']['dataset_version']}, model_version={model_test_record['params']['model_version']}, threshold={model_test_record['params']['operating_threshold']}",
            "qa_judgment": "test 평가 조건과 metric이 함께 기록되었습니다." if record_ready else "평가 기록 필드 누락을 확인해야 합니다.",
            "owner": "QA/ML Engineering",
            "next_action": "새 모델 또는 새 데이터와 비교할 때 params를 먼저 맞춥니다.",
        },
        {
            "evidence": "test_error_profile",
            "observed": f"precision={model_test_record['metrics']['precision']:.4f}, recall={model_test_record['metrics']['recall']:.4f}, FP={model_test_record['metrics']['false_positive']:.0f}, FN={model_test_record['metrics']['false_negative']:.0f}",
            "qa_judgment": "Accuracy 단독 확인이 아니라 오류 유형을 함께 봅니다.",
            "owner": "QA/Business Owner",
            "next_action": "업무상 더 위험한 오류 유형과 threshold 기준을 확인합니다.",
        },
        {
            "evidence": "validation_degradation_comparison",
            "observed": f"precision_delta={comparison_deltas['precision']}, FP_delta={comparison_deltas['false_positive']}",
            "qa_judgment": "같은 모델과 threshold 조건에서 입력 품질 저하를 강한 원인 후보로 남깁니다." if condition_ready else "조건 불일치를 먼저 해소해야 합니다.",
            "owner": "QA/Data Engineering",
            "next_action": "3장에서 serving 응답의 model_version, threshold, feature order를 확인합니다.",
        },
    ]
)
report_sentence = (
    f"model_test_eval 실행 기록은 dataset_version={model_test_record['params']['dataset_version']}, "
    f"model_version={model_test_record['params']['model_version']}, "
    f"operating_threshold={model_test_record['params']['operating_threshold']} 조건에서 계산되었습니다. "
    f"Precision은 {model_test_record['metrics']['precision']:.4f}, Recall은 {model_test_record['metrics']['recall']:.4f}, "
    f"FP는 {model_test_record['metrics']['false_positive']:.0f}건, FN은 {model_test_record['metrics']['false_negative']:.0f}건입니다. "
    f"validation 품질 저하 비교에서는 같은 모델과 threshold 조건에서 Precision delta={comparison_deltas['precision']}와 "
    f"FP delta={comparison_deltas['false_positive']}가 관측되었으므로, 3장에서 API의 model_version과 threshold 일치성을 확인합니다."
)

display(chapter_02_tracking_packet)
print(report_sentence)

### 5-2. 실패 시 확인 포인트

실행이 실패하면 먼저 `TEST_DATASET_PATH`와 `chapter_02_baseline.pkl` 존재 여부를 확인합니다. 데이터와 모델 artifact가 없으면 `labs/prepare_data.py`, `10_train_baseline.py`, `11_evaluate_and_record.py` 실행 순서를 먼저 점검해야 합니다.

JSON 기록이 생성되지 않으면 `artifacts/experiments/chapter_02` 쓰기 권한과 경로를 확인합니다. 이 notebook은 일반 로컬 재생성 경로이므로 JupyterLite에서 실행할 대상이 아닙니다.

MLflow 기록이 없으면 먼저 `mlflow_available` 값을 봅니다. MLflow package가 설치되어 있지 않으면 정상적으로 JSON artifact만 사용해도 됩니다. package가 있는데 tracking DB가 남지 않으면 `artifacts/mlflow.db` 경로와 MLflow tracking URI 설정을 확인합니다.